# Modelos de IA do VisionHat

Este notebook apresenta três modelos de Machine Learning para a disciplina Artificial Intelligence & Chatbot: K-Means, Random Forest e Regressão Linear.

## Origem do dataset

O CSV contém 45 registros artificiais: 15 exemplos por nível de atenção (BAIXO, MEDIO e ALTO). Ele foi elaborado como uma representação simulada do tipo de registro que poderia ser cadastrado por pessoas usuárias do VisionHat no futuro. Os valores foram construídos para o protótipo e não foram coletados de pessoas reais, dos sensores dos óculos ou de ocorrências de campo.

A base permite demonstrar o processo de preparação, treinamento e avaliação dos modelos. As métricas obtidas descrevem somente esse conjunto artificial e não comprovam desempenho em situações reais, nem garantem segurança.

As entradas utilizadas são quantidade de obstáculos, distância média, luminosidade, velocidade de movimento e tipo de obstáculo. O valor de confiança entre 0 e 100 também é um valor simulado definido para o dataset.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

DIRETORIO = Path.cwd()
if not (DIRETORIO / "dataset" / "dados_visionhat.csv").is_file():
    DIRETORIO = DIRETORIO / "ia"

ARQUIVO_DADOS = DIRETORIO / "dataset" / "dados_visionhat.csv"
ARQUIVO_MODELOS = DIRETORIO / "modelos_visionhat.joblib"

FEATURES = [
    "quantidade_obstaculos",
    "distancia_media_metros",
    "luminosidade",
    "velocidade_movimento",
    "tipo_obstaculo_codigo",
]
FEATURES_CLUSTER = ["quantidade_obstaculos", "distancia_media_metros"]
COLUNA_ATENCAO = "nivel_atencao"
COLUNA_CONFIANCA = "confianca_deteccao"
CLASSES_ATENCAO = ["BAIXO", "MEDIO", "ALTO"]

if not ARQUIVO_DADOS.is_file():
    raise FileNotFoundError("Dataset não encontrado: " + str(ARQUIVO_DADOS))

dados = pd.read_csv(ARQUIVO_DADOS)
colunas_necessarias = FEATURES + [COLUNA_ATENCAO, COLUNA_CONFIANCA]
for coluna in colunas_necessarias:
    if coluna not in dados.columns:
        raise ValueError("Coluna ausente no dataset: " + coluna)

if dados[colunas_necessarias].isnull().any().any():
    raise ValueError("O dataset possui valores ausentes.")

contagem_classes = dados[COLUNA_ATENCAO].value_counts()
for classe in CLASSES_ATENCAO:
    if contagem_classes.get(classe, 0) < 5:
        raise ValueError("A classe " + classe + " precisa ter pelo menos cinco registros.")

print("Registros:", len(dados))
print("Distribuição do nível de atenção:")
print(dados[COLUNA_ATENCAO].value_counts().sort_index())
print("Primeiros registros:")
print(dados.head().to_string(index=False))

Registros: 45
Distribuição do nível de atenção:
nivel_atencao
ALTO     15
BAIXO    15
MEDIO    15
Name: count, dtype: int64
Primeiros registros:
 id_registro  quantidade_obstaculos  distancia_media_metros  luminosidade  velocidade_movimento  tipo_obstaculo_codigo nivel_atencao  confianca_deteccao
           1                      1                     8.0            90                   0.4                      1         BAIXO                 58
           2                      1                     7.0            75                   0.8                      2         BAIXO                 62
           3                      2                     6.5            65                   1.0                      3         BAIXO                 64
           4                      1                     5.5            55                   1.2                      4         BAIXO                 67
           5                      2                     8.5           100                   0.

## Modelo 1 - K-Means

O K-Means agrupa registros com características numéricas semelhantes. O nível de atenção não participa do treinamento; é consultado somente depois para interpretar os grupos.

O StandardScaler padroniza as duas variáveis de agrupamento (quantidade de obstáculos e distância média) para que suas escalas sejam comparáveis. Silhueta avalia a separação dos grupos. Adjusted Rand Index compara os agrupamentos com os rótulos artificiais do CSV; não é uma acurácia de previsão.

In [2]:
dados_cluster = dados[FEATURES_CLUSTER]
escalonador = StandardScaler()
dados_escalonados = escalonador.fit_transform(dados_cluster)

modelo_cluster = KMeans(n_clusters=3, n_init=10, random_state=42)
grupos = modelo_cluster.fit_predict(dados_escalonados)

silhueta = silhouette_score(dados_escalonados, grupos)
ari = adjusted_rand_score(dados[COLUNA_ATENCAO], grupos)

comparacao = pd.crosstab(
    pd.Series(grupos, name="grupo"),
    dados[COLUNA_ATENCAO],
)

mapa_niveis = {}
for grupo in comparacao.index:
    mapa_niveis[int(grupo)] = str(comparacao.loc[grupo].idxmax())

print("Silhueta:", round(silhueta, 3))
print("Adjusted Rand Index:", round(ari, 3))
print("Tabela de comparação:")
print(comparacao)
print("Nível representativo de cada grupo:", mapa_niveis)

Silhueta: 0.48
Adjusted Rand Index: 0.807
Tabela de comparação:
nivel_atencao  ALTO  BAIXO  MEDIO
grupo
0                15      0      2
1                 0     14      0
2                 0      1     13
Nível representativo de cada grupo: {0: 'ALTO', 1: 'BAIXO', 2: 'MEDIO'}

## Modelo 2 - Random Forest

A Random Forest classifica o nível de atenção. A avaliação utiliza validação cruzada estratificada em cinco partes (StratifiedKFold). Assim, cada registro recebe uma previsão feita por um modelo que não utilizou aquele registro no respectivo treino.

Após calcular as métricas com as previsões out-of-fold, o modelo final é treinado novamente com todos os registros disponíveis para ser exportado e utilizado pela API.

In [3]:
x_classificacao = dados[FEATURES]
y_classificacao = dados[COLUNA_ATENCAO]

validacao_classificacao = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)
modelo_classificacao = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
)

previsoes_classificacao = cross_val_predict(
    modelo_classificacao,
    x_classificacao,
    y_classificacao,
    cv=validacao_classificacao,
)

acuracia = accuracy_score(y_classificacao, previsoes_classificacao)
f1_macro = f1_score(
    y_classificacao,
    previsoes_classificacao,
    labels=CLASSES_ATENCAO,
    average="macro",
    zero_division=0,
)

print("Acurácia:", round(acuracia, 3))
print("F1 macro:", round(f1_macro, 3))
print("Relatório de classificação:")
print(classification_report(
    y_classificacao,
    previsoes_classificacao,
    labels=CLASSES_ATENCAO,
    zero_division=0,
))
matriz = confusion_matrix(
    y_classificacao,
    previsoes_classificacao,
    labels=CLASSES_ATENCAO,
)
print("Matriz de confusão:")
print(pd.DataFrame(
    matriz,
    index=CLASSES_ATENCAO,
    columns=CLASSES_ATENCAO,
))

modelo_classificacao.fit(x_classificacao, y_classificacao)

Acurácia: 0.956
F1 macro: 0.955
Relatório de classificação:
              precision    recall  f1-score   support

       BAIXO       1.00      1.00      1.00        15
       MEDIO       1.00      0.87      0.93        15
        ALTO       0.88      1.00      0.94        15

    accuracy                           0.96        45
   macro avg       0.96      0.96      0.96        45
weighted avg       0.96      0.96      0.96        45

Matriz de confusão:
       BAIXO  MEDIO  ALTO
BAIXO     15      0     0
MEDIO      0     13     2
ALTO       0      0    15

## Modelo 3 - Regressão Linear

A Regressão Linear estima o valor de confiança definido artificialmente no CSV, em uma escala de 0 a 100. MAE e RMSE medem o erro médio das previsões. R² indica quanto da variação dos valores desta base foi explicada pelo modelo.

As métricas são calculadas com validação cruzada em cinco partes. Depois, a regressão final é treinada com todos os registros. A estimativa é experimental e não corresponde à confiança calculada por um sistema real de visão computacional.

In [4]:
x_regressao = dados[FEATURES]
y_regressao = dados[COLUNA_CONFIANCA]

validacao_regressao = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)
modelo_regressao = LinearRegression()

previsoes_regressao = cross_val_predict(
    modelo_regressao,
    x_regressao,
    y_regressao,
    cv=validacao_regressao,
)

mae = mean_absolute_error(y_regressao, previsoes_regressao)
rmse = mean_squared_error(y_regressao, previsoes_regressao) ** 0.5
r2 = r2_score(y_regressao, previsoes_regressao)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 3))

modelo_regressao.fit(x_regressao, y_regressao)

MAE: 2.25
RMSE: 2.95
R²: 0.937

## Exportação e limitações

O arquivo joblib reúne os três modelos finais, o escalonador, o mapa dos grupos, a ordem das variáveis e as métricas. Random Forest e Regressão Linear são treinadas com todos os registros somente depois de a avaliação por validação cruzada ser concluída.

As métricas de classificação e regressão são estimadas por validação cruzada. As métricas do K-Means são calculadas na própria base utilizada para formar os grupos. Em todos os casos, o CSV é artificial e contém somente 45 registros. Os resultados não devem ser interpretados como desempenho comprovado com pessoas reais, sensores ou situações de mobilidade.

In [5]:
pacote_modelos = {
    "modelo_cluster": modelo_cluster,
    "escalonador": escalonador,
    "mapa_niveis": mapa_niveis,
    "modelo_classificacao": modelo_classificacao,
    "modelo_regressao": modelo_regressao,
    "features": FEATURES,
    "features_cluster": FEATURES_CLUSTER,
    "classes_atencao": [str(classe) for classe in modelo_classificacao.classes_],
    "metrica_silhueta": float(silhueta),
    "metrica_ari": float(ari),
    "metrica_acuracia": float(acuracia),
    "metrica_f1_macro": float(f1_macro),
    "metrica_mae": float(mae),
    "metrica_rmse": float(rmse),
    "metrica_r2": float(r2),
    "metodo_avaliacao": "validacao cruzada de 5 partes",
    "quantidade_registros": int(len(dados)),
    "base_experimental": True,
}

joblib.dump(pacote_modelos, ARQUIVO_MODELOS)
print("Modelos salvos em:", ARQUIVO_MODELOS)

Modelos salvos em: modelos_visionhat.joblib

## Resumo dos resultados desta base

| Modelo | Métrica | Resultado |
|---|---|---:|
| K-Means | Silhueta | 0,480 |
| K-Means | Adjusted Rand Index | 0,807 |
| Random Forest | Acurácia | 95,6% |
| Random Forest | F1 macro | 0,955 |
| Regressão Linear | MAE | 2,25 pontos |
| Regressão Linear | RMSE | 2,95 pontos |
| Regressão Linear | R² | 0,937 |

Esses números são resultados reproduzíveis para o CSV atual. Não são um teste independente de campo nem evidência de que o VisionHat detecta obstáculos com essa precisão.